# Data Cleaning & Microstructure Filtering

This notebook loads the consolidated dataset (`InputData_v0.01.parquet`), inspects data types, verifies missing values, detects duplicates, validates date ranges, and handles numerical anomalies.

**Author:** Sérgio da Cunha Sobral  
**Date:** June 2025


### 1. Library Imports


In [1]:
import pandas as pd
import numpy as np

### 2. Core Paths & Configuration


In [2]:
DIR_INPUT = "02_data/raw/"
FICHEIRO_INPUT = f"{DIR_INPUT}InputData_v0.01.parquet"
FICHEIRO_OUTPUT = f"{DIR_INPUT}InputData_v0.02.parquet"


### 3. Data Ingestion


In [3]:
# Load parquet file
df = pd.read_parquet(FICHEIRO_INPUT)

# Visão geral
print("Shape:", df.shape)
df.head()

Shape: (15144974, 26)


,underlying_symbol,quote_date,root,expiration,strike,option_type,open,high,low,close,...,underlying_ask_1545,bid_size_eod,bid_eod,ask_size_eod,ask_eod,underlying_bid_eod,underlying_ask_eod,vwap,open_interest,delivery_code
0,^SPX,2009-01-02,JXA,2009-01-02,750.0,c,0.0,0.0,0.0,0.0,...,933.41,0,0.0,0,0.0,931.8,931.8,0.0,0,0
1,^SPX,2009-01-02,JXA,2009-01-02,750.0,p,0.0,0.0,0.0,0.0,...,933.41,0,0.0,0,0.0,931.8,931.8,0.0,1751,0
2,^SPX,2009-01-02,JXA,2009-01-02,775.0,c,0.0,0.0,0.0,0.0,...,933.41,0,0.0,0,0.0,931.8,931.8,0.0,0,0
3,^SPX,2009-01-02,JXA,2009-01-02,775.0,p,0.0,0.0,0.0,0.0,...,933.41,0,0.0,0,0.0,931.8,931.8,0.0,458,0
4,^SPX,2009-01-02,JXA,2009-01-02,800.0,c,0.0,0.0,0.0,0.0,...,933.41,0,0.0,0,0.0,931.8,931.8,0.0,0,0


### 4. Overall Dataset Structure


#### Structural Overview & Shape


In [4]:
# General structure
print("Shape do dataset:", df.shape)

# Column names, schemas, and unique value counts
print("\nColumns and data schemas:")
display(pd.DataFrame({'Tipo': df.dtypes, 'Nº de únicos': df.nunique()}))

# Duplicate column names
print("\nDuplicate column names:")
print(df.columns[df.columns.duplicated()].tolist())

# Memory usage
print("\nTotal memory usage (MB):")
print(round(df.memory_usage(deep=True).sum() / (1024 ** 2), 2), "MB")

# Unique values in primary categorical variables
print("\nValores únicos de 'option_type':")
print(df["option_type"].unique())

print("\nValores únicos de 'delivery_code':")
print(df["delivery_code"].unique())

# First rows
df.head()


Shape do dataset: (15144974, 26)

Colunas e tipos de dados:


,Tipo,Nº de únicos
underlying_symbol,object,1
quote_date,object,2597
root,object,45
expiration,object,915
strike,float64,586
option_type,object,4
open,float64,56280
high,float64,56407
low,float64,56079
close,float64,56249



Nomes de colunas duplicados:
[]

Uso de memória total (MB):
6988.21 MB

Valores únicos de 'option_type':
['c' 'p' 'C' 'P']

Valores únicos de 'delivery_code':
['0' '$100' '0.0' 'nan']


,underlying_symbol,quote_date,root,expiration,strike,option_type,open,high,low,close,...,underlying_ask_1545,bid_size_eod,bid_eod,ask_size_eod,ask_eod,underlying_bid_eod,underlying_ask_eod,vwap,open_interest,delivery_code
0,^SPX,2009-01-02,JXA,2009-01-02,750.0,c,0.0,0.0,0.0,0.0,...,933.41,0,0.0,0,0.0,931.8,931.8,0.0,0,0
1,^SPX,2009-01-02,JXA,2009-01-02,750.0,p,0.0,0.0,0.0,0.0,...,933.41,0,0.0,0,0.0,931.8,931.8,0.0,1751,0
2,^SPX,2009-01-02,JXA,2009-01-02,775.0,c,0.0,0.0,0.0,0.0,...,933.41,0,0.0,0,0.0,931.8,931.8,0.0,0,0
3,^SPX,2009-01-02,JXA,2009-01-02,775.0,p,0.0,0.0,0.0,0.0,...,933.41,0,0.0,0,0.0,931.8,931.8,0.0,458,0
4,^SPX,2009-01-02,JXA,2009-01-02,800.0,c,0.0,0.0,0.0,0.0,...,933.41,0,0.0,0,0.0,931.8,931.8,0.0,0,0


### 5. Data Type Conversion & Integrity


In [5]:
# Create working copy for editing
df_1 = df.copy()

# Standardize option_type to uppercase
df_1["option_type"] = df_1["option_type"].str.upper()

# Drop delivery_code column
if "delivery_code" in df_1.columns:
    df_1.drop(columns=["delivery_code"], inplace=True)

# Expected schema per column
tipos_esperados = {
    "underlying_symbol": "string",
    "quote_date": "datetime64[ns]",
    "root": "string",
    "expiration": "datetime64[ns]",
    "strike": "float64",
    "option_type": "category",
    
    "open": "float64",
    "high": "float64",
    "low": "float64",
    "close": "float64",
    
    "trade_volume": "Int64",
    
    "bid_size_1545": "Int64",
    "ask_size_1545": "Int64",
    "bid_1545": "float64",
    "ask_1545": "float64",
    
    "underlying_bid_1545": "float64",
    "underlying_ask_1545": "float64",
    
    "bid_size_eod": "Int64",
    "ask_size_eod": "Int64",
    "bid_eod": "float64",
    "ask_eod": "float64",
    
    "underlying_bid_eod": "float64",
    "underlying_ask_eod": "float64",
    
    "vwap": "float64",
    "open_interest": "Int64"
}

# Apply controlled data type conversions
for col, tipo in tipos_esperados.items():
    if col in df_1.columns:
        try:
            df_1[col] = df_1[col].astype(tipo)
        except Exception as e:
            print(f"❌ Error converting column '{col}' to {tipo}: {e}")

# Check for final discrepancies
print("\n🔍 Schema comparison (actual vs expected types):")
comparacao_tipos = pd.DataFrame({
    "Tipo Atual": df_1.dtypes,
    "Expected Type": pd.Series(tipos_esperados)
}).dropna()

comparacao_tipos["Match"] = comparacao_tipos["Tipo Atual"].astype(str) == comparacao_tipos["Expected Type"]
display(comparacao_tipos)

df_1.sample(20)



🔍 Comparação de tipos finais vs esperados:


,Tipo Atual,Tipo Esperado,Match
ask_1545,float64,float64,True
ask_eod,float64,float64,True
ask_size_1545,Int64,Int64,True
ask_size_eod,Int64,Int64,True
bid_1545,float64,float64,True
bid_eod,float64,float64,True
bid_size_1545,Int64,Int64,True
bid_size_eod,Int64,Int64,True
close,float64,float64,True
expiration,datetime64[ns],datetime64[ns],True


,underlying_symbol,quote_date,root,expiration,strike,option_type,open,high,low,close,...,underlying_bid_1545,underlying_ask_1545,bid_size_eod,bid_eod,ask_size_eod,ask_eod,underlying_bid_eod,underlying_ask_eod,vwap,open_interest
6637374,^SPX,2016-02-12,SPXW,2016-02-12,1940.0,C,0.05,0.05,0.05,0.05,...,1862.02,1862.02,0,0.00,3000,0.05,1864.78,1864.78,0.05,3507
9566331,^SPX,2017-07-21,SPXW,2017-09-29,2535.0,C,0.00,0.00,0.00,0.00,...,2469.86,2469.86,566,7.90,377,8.40,2472.54,2472.54,0.00,252
5834270,^SPX,2015-09-17,SPXW,2015-10-23,1400.0,C,0.00,0.00,0.00,0.00,...,1995.09,1995.09,1,571.50,31,595.00,1990.20,1990.20,0.00,0
3503558,^SPX,2014-04-30,SPXW,2014-06-06,1855.0,C,0.00,0.00,0.00,0.00,...,1884.66,1884.66,20,42.80,263,44.70,1883.95,1883.95,0.00,1
1077217,^SPX,2011-01-12,SPX,2011-02-19,1260.0,P,16.20,16.60,14.80,14.90,...,1283.85,1283.85,33,13.50,33,15.50,1285.96,1285.96,16.07,7722
14722614,^SPX,2019-03-15,SPXW,2019-05-03,2475.0,C,0.00,0.00,0.00,0.00,...,2823.20,2823.79,20,355.50,20,358.20,2787.85,2845.05,0.00,0
9362375,^SPX,2017-06-22,SPXW,2017-06-30,2065.0,C,0.00,0.00,0.00,0.00,...,2435.88,2435.88,70,364.00,70,373.10,2434.50,2434.50,0.00,1
6654745,^SPX,2016-02-17,SPX,2016-03-18,2000.0,P,100.50,100.50,85.28,87.85,...,1926.58,1926.58,20,86.50,9,89.40,1926.82,1926.82,91.32,95691
4753800,^SPX,2015-03-10,SPXW,2015-04-02,1815.0,C,0.00,0.00,0.00,0.00,...,2049.80,2049.80,24,223.20,100,232.30,2044.16,2044.16,0.00,0
7261437,^SPX,2016-06-14,SPX,2017-03-17,500.0,C,0.00,0.00,0.00,0.00,...,2075.09,2075.09,12,1541.00,2,1544.60,2075.32,2075.32,0.00,0


### 6. Descriptive Summary Analysis


In [6]:
print("🔍 Detailed Descriptive Analysis\n")

# 1. Missing values audit
print("🟠 Missing values count per column:")
missing = df_1.isnull().sum()
missing_percent = (missing / len(df_1)) * 100
missing_table = pd.DataFrame({
    "Missing": missing,
    "Percent (%)": missing_percent.round(2)
})
display(missing_table[missing_table["Missing"] > 0].sort_values("Percent (%)", ascending=False))

# 2. Duplicates (identical complete rows)
print("\n🟡 Duplicate row count:", df_1.duplicated().sum())

# 3. Categorical variables: unique counts and top categories
print("\n🔠 Categorical variables - Unique category counts:")
cat_cols = df_1.select_dtypes(include=["object", "category"]).columns
for col in cat_cols:
    print(f"\n🧷 {col} ({df_1[col].nunique()} unique categories)")
    print(df_1[col].value_counts().head(10))

# 4. Invariant columns (single unique value) -> exclusion candidates
print("\n🔴 Invariant columns (single unique value):")
for col in df_1.columns:
    if df_1[col].nunique() == 1:
        print(f" - {col}")

# 5. Near-constant columns (>95% identical)
print("\n🟡 Near-constant columns (>95% identical):")
for col in df_1.columns:
    top_freq = df_1[col].value_counts(normalize=True, dropna=False).values[0]
    if top_freq > 0.95:
        print(f" - {col} ({top_freq:.2%} igual)")

# 6. General statistics (continuous and categorical)
print("\n📊 Summary statistics (continuous):")
display(df_1.describe())

print("\n📊 Summary statistics (categorical):")
display(df_1.describe(include=["object", "category"]))


🔍 Análise Descritiva Detalhada

🟠 Valores em falta por coluna:


,Missing,Percent (%)



🟡 Número de linhas duplicadas: 0

🔠 Variáveis categóricas - Nº de categorias únicas:

🧷 option_type (2 categorias únicas)
option_type
C    7572494
P    7572480
Name: count, dtype: int64

🔴 Colunas com apenas 1 valor único:
 - underlying_symbol

🟡 Colunas quase constantes (>95% igual):
 - underlying_symbol (100.00% igual)

📊 Estatísticas resumo (contínuas):


,quote_date,expiration,strike,open,high,low,close,trade_volume,bid_size_1545,bid_1545,...,underlying_bid_1545,underlying_ask_1545,bid_size_eod,bid_eod,ask_size_eod,ask_eod,underlying_bid_eod,underlying_ask_eod,vwap,open_interest
count,15144974,15144974,1.514497e+07,1.514497e+07,1.514497e+07,1.514497e+07,1.514497e+07,15144974.0,15144974.0,1.514497e+07,...,1.514497e+07,1.514497e+07,15144974.0,1.514497e+07,15144974.0,1.514497e+07,1.514497e+07,1.514497e+07,1.514497e+07,15144974.0
mean,2016-01-09 00:53:30.303193088,2016-05-10 08:12:02.009346816,1.890139e+03,9.951283e+00,1.027256e+01,9.674647e+00,9.956508e+00,156.730978,142.282429,2.044039e+02,...,2.163320e+03,2.163377e+03,117.286434,2.042570e+02,158.888095,2.078411e+02,2.162091e+03,2.168235e+03,2.596672e+105,2298.310953
min,2009-01-02 00:00:00,2009-01-02 00:00:00,5.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00,...,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.0,0.000000e+00,6.765300e+02,6.765300e+02,-1.019863e+16,0.0
25%,2014-07-25 00:00:00,2014-12-12 00:00:00,1.405000e+03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,12.0,1.700000e+00,...,1.904590e+03,1.904590e+03,8.0,1.650000e+00,11.0,2.150000e+00,1.906130e+03,1.906130e+03,0.000000e+00,0.0
50%,2016-08-16 00:00:00,2016-12-16 00:00:00,1.925000e+03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,64.0,4.950000e+01,...,2.141150e+03,2.141150e+03,44.0,4.930000e+01,60.0,5.170000e+01,2.141340e+03,2.141340e+03,0.000000e+00,15.0
75%,2018-03-20 00:00:00,2018-06-15 00:00:00,2.390000e+03,1.000000e-01,1.500000e-01,1.000000e-01,1.000000e-01,1.0,163.0,2.749000e+02,...,2.656430e+03,2.656430e+03,124.0,2.746000e+02,160.0,2.818000e+02,2.648050e+03,2.663420e+03,1.500000e-01,559.0
max,2019-04-30 00:00:00,2021-12-17 00:00:00,4.200000e+03,4.482300e+03,9.105000e+03,2.823600e+03,2.827530e+03,140501.0,20404.0,2.817400e+03,...,2.945560e+03,2.946090e+03,20404.0,2.820000e+03,33665.0,2.826400e+03,2.930750e+03,2.966990e+03,3.932653e+112,624873.0
std,NaN,NaN,6.696030e+02,5.487207e+01,5.569224e+01,5.463388e+01,5.489316e+01,1159.117227,233.955934,3.279206e+02,...,5.416276e+02,5.416852e+02,203.642348,3.278834e+02,369.944928,3.305750e+02,5.339517e+02,5.400279e+02,1.010535e+109,9281.159596



📊 Estatísticas resumo (categóricas):


,option_type
count,15144974
unique,2
top,C
freq,7572494


##### Summary of Descriptive Quality Checks

- **Missing Values:** Zero missing values detected across all columns.
- **Duplicates:** Zero duplicate records found.
- **Categorical Variables:** `option_type` correctly contains only valid `C` and `P` flags.
- **Low Information Columns:** `underlying_symbol` contains 100% constant values (`^SPX`) and is removed.
- **Continuous Variables at Zero:** Bid and Ask have zero values for non-executable quotes, motivating strict liquidity filtering in subsequent stages.


### 7. Consistency Checks & Expiration Logic


In [7]:
print("🔎 Part 3 — Data Coherence & Sanity Audit\n")

# A. Data types recap
print("📌 Current DataFrame schema:")
display(df_1.dtypes)

# B. Analysis of invalid values in numeric columns
# Columns where negative or zero values may be anomalous
colunas_sem_negativos = [
    "strike", "open", "high", "low", "close",
    "bid_1545", "ask_1545", "bid_eod", "ask_eod",
    "underlying_bid_1545", "underlying_ask_1545",
    "underlying_bid_eod", "underlying_ask_eod",
    "vwap"
]

colunas_sem_zeros = [
    "strike", "bid_1545", "ask_1545", "bid_eod", "ask_eod", "vwap"
]

# Check for negative values
print("\n🔴 Columns containing anomalous negative values:")
for col in colunas_sem_negativos:
    if (df_1[col] < 0).any():
        negativos = (df_1[col] < 0).sum()
        print(f" - {col}: {negativos} negative values")

# Check for zero values
print("\n🟡 Columns with high zero counts (potential anomalies):")
for col in colunas_sem_zeros:
    zeros = (df_1[col] == 0).sum()
    perc = zeros / len(df_1)
    if perc > 0.1:
        print(f" - {col}: {zeros} zeros ({perc:.2%})")

# Check for extreme values in vwap
print("\n❗ Extreme values detected in vwap:")
print("Top 5 highest values:")
display(df_1["vwap"].sort_values(ascending=False).head())

# C. Date analysis
print("\n📆 Temporal column audit (quote_date and expiration):")

# Confirmar tipo datetime
print("Tipos:")
print(f" - quote_date: {df_1['quote_date'].dtype}")
print(f" - expiration: {df_1['expiration'].dtype}")

# Validação: expiration >= quote_date
erros_data = (df_1["expiration"] < df_1["quote_date"]).sum()
print(f"\n🔴 Anomalous rows where expiration < quote_date: {erros_data}")

# Date ranges
print("\nDate range:")
print(f" - quote_date: {df_1['quote_date'].min().date()} → {df_1['quote_date'].max().date()}")
print(f" - expiration: {df_1['expiration'].min().date()} → {df_1['expiration'].max().date()}")

# Unique date counts and yearly breakdown
print("\nNumber of unique dates:")
print(f" - quote_date: {df_1['quote_date'].nunique()}")
print(f" - expiration: {df_1['expiration'].nunique()}")

print("\nTop 5 years by record count (quote_date):")
display(df_1["quote_date"].dt.year.value_counts().sort_index().tail(5))

# Frequency of each option root
print("📊 Unique value distribution for 'root' column:")
display(df_1["root"].value_counts().sort_values(ascending=False))


🔎 Parte 3 — Verificação de Coerência dos Dados

📌 Tipos de dados atuais:


underlying_symbol      string[python]
quote_date             datetime64[ns]
root                   string[python]
expiration             datetime64[ns]
strike                        float64
option_type                  category
open                          float64
high                          float64
low                           float64
close                         float64
trade_volume                    Int64
bid_size_1545                   Int64
bid_1545                      float64
ask_size_1545                   Int64
ask_1545                      float64
underlying_bid_1545           float64
underlying_ask_1545           float64
bid_size_eod                    Int64
bid_eod                       float64
ask_size_eod                    Int64
ask_eod                       float64
underlying_bid_eod            float64
underlying_ask_eod            float64
vwap                          float64
open_interest                   Int64
dtype: object


🔴 Colunas com valores negativos (onde não deveriam existir):
 - vwap: 7 valores negativos

🟡 Colunas com muitos valores zero (onde 0 pode ser suspeito):
 - vwap: 11060019 zeros (73.03%)

❗ Valores extremos em vwap:
Top 5 valores mais altos:


691167    3.932653e+112
737628     1.451398e+93
171562     7.491149e+25
675705     6.831693e+25
170582     6.710365e+25
Name: vwap, dtype: float64


📆 Análise das colunas temporais (quote_date e expiration):
Tipos:
 - quote_date: datetime64[ns]
 - expiration: datetime64[ns]

🔴 Nº de linhas com expiration < quote_date: 0

Intervalo de datas:
 - quote_date: 2009-01-02 → 2019-04-30
 - expiration: 2009-01-02 → 2021-12-17

Nº de datas únicas:
 - quote_date: 2597
 - expiration: 915

Top 5 anos com mais registos (quote_date):


quote_date
2015    1982066
2016    1892673
2017    2395020
2018    3287836
2019    1138855
Name: count, dtype: int64

📊 Contagem de valores únicos na coluna 'root':


root
SPXW     8311267
SPX      5585608
SPXQ      372106
BSZ       170348
SPXPM      59798
SXB        50790
SPZ        49106
SPQ        49096
SYG        43934
SPT        42148
BSK        41106
SRO        35075
SZP        31698
SYU        25829
SZJ        19354
SXY        19118
SZT        18814
SZQ        15823
QSE        15761
SPV        14560
QZQ        14525
SXZ        12452
QSZ        12325
SZD        12216
SZU        12070
BSF        12038
SPL        11890
SXG        11366
SZV         9100
SXM         8582
SYZ         7160
SAQ         6145
SLQ         5994
SVP         5452
SQP         4985
SPB         3608
SXJ         3558
SYV         3498
JXB         2842
VSTRP       2782
SQG         2760
JXD         2732
JXA         2506
SKQ         1999
JXE         1050
Name: count, dtype: Int64

##### Summary of Consistency Verification

- **Data Types:** Validated and strictly consistent with domain schema.
- **Dates:** `quote_date` and `expiration` correctly parsed as datetimes. Zero occurrences of `expiration < quote_date`.
- **Date Range:** `quote_date` spans 2009-01-02 to 2019-04-30; `expiration` spans up to 2021-12-17 (long-term LEAPS).


### 8. Export Cleaned Parquet Dataset


In [8]:
# Subset of relevant columns
df_final = df_1[[
    "quote_date",
    "root",
    "expiration",
    "strike",
    "option_type",
    "bid_size_1545", "ask_size_1545",
    "bid_1545", "ask_1545",
    "underlying_bid_1545", "underlying_ask_1545",
    "bid_size_eod", "ask_size_eod",
    "bid_eod", "ask_eod",
    "underlying_bid_eod", "underlying_ask_eod",
    "open_interest"
]]

# Save figures/data as Parquet
df_final.to_parquet(FICHEIRO_OUTPUT, index=False)
print("Clean Parquet file saved successfully:", FICHEIRO_OUTPUT)


✅ Ficheiro salvo em formato Parquet: 02. Dados/01. Inputs/InputData_v0.02.parquet
